# Nivel 1

## Parte A

### Carregamento

In [190]:
import pandas as pd
import json

with open("../dados/dados_nivel_1.json", "r") as f:
    dados_1 = json.load(f)

taxa_cambio = dados_1["taxa_cambio_usd_brl"]
df = pd.json_normalize(dados_1["operacoes"])
print("Dataframe: ")
df.head(10)

Dataframe: 


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


### Limpeza

Como o json era pequeno, li cada um dos elementos em busca de qualquer ponto "diferente", o que eu relatei a seguir é o que foi encontrado

**1. `OP-0007` duplicada -> removida com `drop_duplicates()`**

São 20 registros para 19 `id` distintos. As duas linhas de `OP-0007` são idênticas nos 9 campos.

O que decide o tratamento aqui é o `id`, não os valores. `id` é a chave primária da operação, então duas linhas com o mesmo `id` são o mesmo evento gravado duas vezes e não duas transações parecidas. Desse modo, a melhor opção é remover esse ponto


**2. `OP-0017` sem data -> mantida, sem imputar e sem dropar**

Uma linha tem `data: null`, com `observacao: "data nao capturada pelo sistema"`.

*Por que não dropar.* A ausência da data é informação, não vazio. Ela registra uma falha de captura na origem, e metadado faltando é por si só um indicador de anomalia que vale acompanhar. Por exemplo, um ponto em que não tem data registrada e que a transação foi feita em espécie é algo a se notar no contexto de PLD

*Por que não imputar.* Qualquer preenchimento (moda, forward fill, data vizinha do mesmo cliente) inventa uma posição no tempo que o dado não tem. 

In [191]:
df_clean = df.drop_duplicates().reset_index(drop=True)
print("Df sem duplicata: ")
df_clean.head(10)

Df sem duplicata: 


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,
9,OP-0010,CLI-A-4,2026-03-03,3800,BRL,cartao,pagamento,Alfa Comercio LTDA,


### Normalização -> 1 único ponto que foi necessário converter de USD -> BRL

In [192]:
df_clean["valor"] = df_clean["valor"].astype(float) # necessário converter tudo para float
mask = df_clean["moeda"] == "USD"
df_clean.loc[mask, "valor"] *= taxa_cambio
df_clean.loc[mask, "moeda"] = "BRL"

df_clean

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100.0,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300.0,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800.0,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300.0,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900.0,BRL,ted,transferencia_enviada,Delta Transportes,
5,OP-0006,CLI-A-2,2026-03-14,27000.0,BRL,ted,transferencia_enviada,Delta Transportes,
6,OP-0007,CLI-A-3,2026-03-05,17200.0,BRL,pix,transferencia_enviada,Epsilon Consultoria,
7,OP-0008,CLI-A-3,2026-03-05,15200.0,BRL,pix,transferencia_enviada,Epsilon Consultoria,
8,OP-0009,CLI-A-3,2026-03-05,16100.0,BRL,pix,transferencia_enviada,Zeta Importacao,
9,OP-0010,CLI-A-4,2026-03-03,3800.0,BRL,cartao,pagamento,Alfa Comercio LTDA,


### Agregações

volume total transacionado por cliente

In [193]:
vol_total_cliente = df_clean.groupby("cliente_id").agg(
    volume_transacionado=("valor", "sum")
)
vol_total_cliente

,volume_transacionado
cliente_id,
CLI-A-1,57500.0
CLI-A-2,52900.0
CLI-A-3,48500.0
CLI-A-4,79500.0
CLI-A-5,16900.0
CLI-A-6,10200.0


quantidade de operações por canal

In [194]:
qnt_op_canal = df_clean.groupby("canal").agg(
    operacoes_canal=("canal", "count")
)
qnt_op_canal

,operacoes_canal
canal,
boleto,3
cartao,2
especie,1
pix,8
ted,5


### Regras determinísticas

Regra 1 - Fracionamento. Sinalize o cliente que, em uma mesma data, realizou 3 ou mais operações cuja soma ultrapassa R$ 50.000,00, sendo que nenhuma operação isolada atinge R$ 20.000,00.

Flags por grupo

In [195]:
lim_soma = 50000      # "soma ultrapassa R$ 50.000,00"
lim_op = 20000  # "nenhuma operação isolada atinge R$ 20.000,00"
min_op = 3

transacoes_cliente_dia = df_clean.groupby(["cliente_id", "data"], dropna=True).agg( # dropna para o caso em que falta data
    n_operacoes=("valor", "count"),
    soma_dia=("valor", "sum"),
    maior_operacao=("valor", "max"),
)

transacoes_cliente_dia["flag_fracionamento"] = (
    (transacoes_cliente_dia["n_operacoes"] >= min_op)
    & (transacoes_cliente_dia["soma_dia"] > lim_soma)
    & (transacoes_cliente_dia["maior_operacao"] < lim_op)
)
transacoes_cliente_dia

n_operacoes  soma_dia  maior_operacao  \
cliente_id data                                                
CLI-A-1    2026-03-09            3   54200.0         18800.0   
           2026-03-21            1    3300.0          3300.0   
CLI-A-2    2026-03-14            2   52900.0         27000.0   
CLI-A-3    2026-03-05            3   48500.0         17200.0   
CLI-A-4    2026-03-03            1    3800.0          3800.0   
           2026-03-11            1    5100.0          5100.0   
           2026-03-18            1    5800.0          5800.0   
           2026-03-24            1   64800.0         64800.0   
CLI-A-5    2026-03-07            1    2900.0          2900.0   
           2026-03-16            1    7000.0          7000.0   
           2026-03-26            1    2700.0          2700.0   
CLI-A-6    2026-03-12            1    8800.0          8800.0   
           2026-03-28            1    1400.0          1400.0   

                       flag_fracionamento  
cliente_id data                            
CLI-A-1    2026-03-09                True  
           2026-03-21               False  
CLI-A-2    2026-03-14               False  
CLI-A-3    2026-03-05               False  
CLI-A-4    2026-03-03               False  
           2026-03-11               False  
           2026-03-18               False  
           2026-03-24               False  
CLI-A-5    2026-03-07               False  
           2026-03-16               False  
           2026-03-26               False  
CLI-A-6    2026-03-12               False  
           2026-03-28               False

Adicionando flag ao *cliente*

In [196]:
df_clean["flag_fracionamento"] = (
    pd.MultiIndex.from_frame(df_clean[["cliente_id", "data"]])
    .map(transacoes_cliente_dia["flag_fracionamento"])
    .fillna(False)  # operação sem data não tem par correspondente -> não sinalizada
    .astype(bool)
)

df_clean["flag_regra1_fracionamento"] = (
    df_clean.groupby("cliente_id")["flag_fracionamento"].transform("any")
)

df_clean = df_clean.drop(columns="flag_fracionamento")
df_clean[["id", "cliente_id", "data", "valor", "flag_regra1_fracionamento"]]

,id,cliente_id,data,valor,flag_regra1_fracionamento
0,OP-0001,CLI-A-1,2026-03-09,18100.0,True
1,OP-0002,CLI-A-1,2026-03-09,17300.0,True
2,OP-0003,CLI-A-1,2026-03-09,18800.0,True
3,OP-0004,CLI-A-1,2026-03-21,3300.0,True
4,OP-0005,CLI-A-2,2026-03-14,25900.0,False
5,OP-0006,CLI-A-2,2026-03-14,27000.0,False
6,OP-0007,CLI-A-3,2026-03-05,17200.0,False
7,OP-0008,CLI-A-3,2026-03-05,15200.0,False
8,OP-0009,CLI-A-3,2026-03-05,16100.0,False
9,OP-0010,CLI-A-4,2026-03-03,3800.0,False


Regra 2 - Valor atípico. Sinalize a operação cujo valor em BRL seja superior a 5× a mediana dos valores daquele mesmo cliente. Aplique apenas a clientes com 4 ou mais operações. 

In [197]:
lim_mediana = 5 
min_op_r2 = 4

transacoes_cliente_mediana = df_clean.groupby(["cliente_id"]).agg(
    n_operacoes=("valor", "count"),
    mediana=("valor", "median"),
)
transacoes_cliente_mediana

,n_operacoes,mediana
cliente_id,,
CLI-A-1,4,17700.0
CLI-A-2,2,26450.0
CLI-A-3,3,16100.0
CLI-A-4,4,5450.0
CLI-A-5,4,3600.0
CLI-A-6,2,5100.0


In [198]:
mediana_cliente = df_clean["cliente_id"].map(transacoes_cliente_mediana["mediana"])
n_op_cliente = df_clean["cliente_id"].map(transacoes_cliente_mediana["n_operacoes"])

df_clean["flag_regra2_valor_atipico"] = (
    (n_op_cliente >= min_op_r2)
    & (df_clean["valor"] > lim_mediana * mediana_cliente)
)

df_clean[["id", "cliente_id", "valor", "flag_regra1_fracionamento", "flag_regra2_valor_atipico"]]

,id,cliente_id,valor,flag_regra1_fracionamento,flag_regra2_valor_atipico
0,OP-0001,CLI-A-1,18100.0,True,False
1,OP-0002,CLI-A-1,17300.0,True,False
2,OP-0003,CLI-A-1,18800.0,True,False
3,OP-0004,CLI-A-1,3300.0,True,False
4,OP-0005,CLI-A-2,25900.0,False,False
5,OP-0006,CLI-A-2,27000.0,False,False
6,OP-0007,CLI-A-3,17200.0,False,False
7,OP-0008,CLI-A-3,15200.0,False,False
8,OP-0009,CLI-A-3,16100.0,False,False
9,OP-0010,CLI-A-4,3800.0,False,False


### Validando as regras

Uma coluna por condição. A regra dispara onde as três dão `True`.

- **Captura** `CLI-A-1 / 09-03`: 3 ops, R$ 54.200, maior R$ 18.800.
- **Não captura** `CLI-A-3 / 05-03`: mesmo desenho, mas soma R$ 48.500 — falha só em `c2`, por R$ 1.500.
- **Não captura** `CLI-A-2 / 14-03`: soma passa de 50k, mas são 2 ops e ambas acima de 20k — falha em `c1` e `c3`.

`CLI-A-3` e `CLI-A-2` são parecidos. No primeiro, sem a limpeza ele viraria falso positivo: com a duplicata `OP-0007`, seriam 4 ops somando R$ 65.700. No segundo, passa dos 50k, mas tem apenas 2 operações

In [199]:
validacao_r1 = transacoes_cliente_dia[transacoes_cliente_dia["n_operacoes"] >= 2].copy()
validacao_r1["c1_3_ou_mais_ops"] = validacao_r1["n_operacoes"] >= min_op
validacao_r1["c2_soma_acima_50k"] = validacao_r1["soma_dia"] > lim_soma
validacao_r1["c3_nenhuma_atinge_20k"] = validacao_r1["maior_operacao"] < lim_op
validacao_r1

,,n_operacoes,soma_dia,maior_operacao,flag_fracionamento,c1_3_ou_mais_ops,c2_soma_acima_50k,c3_nenhuma_atinge_20k
cliente_id,data,,,,,,,
CLI-A-1,2026-03-09,3,54200.0,18800.0,True,True,True,True
CLI-A-2,2026-03-14,2,52900.0,27000.0,False,False,True,False
CLI-A-3,2026-03-05,3,48500.0,17200.0,False,True,False,True


Uma coluna por condição, mais a razão `valor / mediana do cliente`.

- **Captura** `OP-0013`: 11,89× a mediana do CLI-A-4.
- **Não captura** `OP-0015`: 1,94×, a maior entre as elegíveis.
- **Não captura** `OP-0018`: 1,73× e cliente com 2 ops — barrado por `c1`.

In [200]:
validacao_r2 = df_clean[["id", "cliente_id", "valor"]].copy()
validacao_r2["mediana_cliente"] = mediana_cliente
validacao_r2["razao"] = (df_clean["valor"] / mediana_cliente).round(2)
validacao_r2["c1_cliente_com_4_ops"] = n_op_cliente >= min_op_r2
validacao_r2["c2_acima_de_5x"] = df_clean["valor"] > lim_mediana * mediana_cliente

validacao_r2.sort_values("razao", ascending=False).head(5)

,id,cliente_id,valor,mediana_cliente,razao,c1_cliente_com_4_ops,c2_acima_de_5x
12,OP-0013,CLI-A-4,64800.0,5450.0,11.89,True,True
14,OP-0015,CLI-A-5,7000.0,3600.0,1.94,True,False
17,OP-0018,CLI-A-6,8800.0,5100.0,1.73,False,False
16,OP-0017,CLI-A-5,4300.0,3600.0,1.19,True,False
6,OP-0007,CLI-A-3,17200.0,16100.0,1.07,False,False


## Parte B

### Escolhendo cliente sinalizado e montando contexto

In [201]:
cliente = df_clean.loc[df_clean["flag_regra1_fracionamento"], "cliente_id"].unique()[0]
ops_cliente = df_clean[df_clean["cliente_id"] == cliente] # operações daquele cliente

dia_flag = transacoes_cliente_dia[transacoes_cliente_dia["flag_fracionamento"]].reset_index()
dia_flag = dia_flag[dia_flag["cliente_id"] == cliente].iloc[0] #dia que causou a flag de fracionamento

dossie = {
    "cliente_id": cliente,
    "janela": f"{ops_cliente['data'].min()} a {ops_cliente['data'].max()}",
    "total_operacoes": int(len(ops_cliente)),
    "volume_total_brl": float(ops_cliente["valor"].sum()),
    "mediana_brl": float(ops_cliente["valor"].median()),
    "media_brl": float(ops_cliente["valor"].mean()),
    "desvio_padrao_brl": float(ops_cliente["valor"].std()),
    "regra_1_fracionamento": {
        "disparou": True,
        "data": str(dia_flag["data"]),
        "operacoes_no_dia": int(dia_flag["n_operacoes"]),
        "soma_no_dia_brl": float(dia_flag["soma_dia"]),
        "maior_operacao_no_dia_brl": float(dia_flag["maior_operacao"]),
        "limites": {"min_operacoes": min_op, "soma_acima_de": lim_soma, "nenhuma_atinge": lim_op},
    },
    "regra_2_valor_atipico": {
        "disparou": bool(ops_cliente["flag_regra2_valor_atipico"].any()),
        "operacoes": ops_cliente.loc[ops_cliente["flag_regra2_valor_atipico"], "id"].tolist(),
    },
    "operacoes": ops_cliente[
        ["id", "data", "valor", "canal", "tipo", "contraparte", "observacao"]
    ].to_dict("records"),
}

print(json.dumps(dossie, ensure_ascii=False, indent=2))

{
  "cliente_id": "CLI-A-1",
  "janela": "2026-03-09 a 2026-03-21",
  "total_operacoes": 4,
  "volume_total_brl": 57500.0,
  "mediana_brl": 17700.0,
  "media_brl": 14375.0,
  "desvio_padrao_brl": 7408.722336993516,
  "regra_1_fracionamento": {
    "disparou": true,
    "data": "2026-03-09",
    "operacoes_no_dia": 3,
    "soma_no_dia_brl": 54200.0,
    "maior_operacao_no_dia_brl": 18800.0,
    "limites": {
      "min_operacoes": 3,
      "soma_acima_de": 50000,
      "nenhuma_atinge": 20000
    }
  },
  "regra_2_valor_atipico": {
    "disparou": false,
    "operacoes": []
  },
  "operacoes": [
    {
      "id": "OP-0001",
      "data": "2026-03-09",
      "valor": 18100.0,
      "canal": "pix",
      "tipo": "transferencia_enviada",
      "contraparte": "Alfa Comercio LTDA",
      "observacao": ""
    },
    {
      "id": "OP-0002",
      "data": "2026-03-09",
      "valor": 17300.0,
      "canal": "pix",
      "tipo": "transferencia_enviada",
      "contraparte": "Alfa Comercio LTDA",

### Montando prompts

* A ideia é montar 1 prompt com uma relação de "risco X retorno" mais conservadora e outra mais arrojada

* llm não pode fazer cálculos sozinha, só interpretação

In [202]:
ESQUEMA = """Responda SOMENTE com um objeto JSON válido, sem nenhum texto fora dele, com exatamente estes campos:
- "nivel_risco": um entre "baixo", "médio", "alto"
- "tipologia_suspeita": string curta nomeando a tipologia
- "red_flags": lista de strings
- "justificativa": string, no máximo 4 frases

Os números do dossiê já foram calculados e conferidos por regras determinísticas. 
Não recalcule somas, não verifique se algum limite foi ultrapassado e não questione se a regra disparou.
Seu trabalho é interpretar e redigir. Em hipótese alguma você deve criar algum cálculo novo ou variável nova interna"""

PROMPT_CONSERVADOR = f"""Você é analista sênior de Prevenção à Lavagem de Dinheiro numa mesa de triagem.

Postura: o custo de um falso positivo é alto. Cada caso escalado consome horas de analista e gera atrito com um cliente possivelmente legítimo. Atribua risco "alto" apenas quando os dados do dossiê sustentarem sozinhos essa conclusão. Na dúvida entre dois níveis, escolha o menor. Não liste como red flag nada que dependa de informação ausente do dossiê, e não trate ausência de informação como indício.

{ESQUEMA}"""

PROMPT_ARROJADO = f"""Você é analista sênior de Prevenção à Lavagem de Dinheiro numa mesa de triagem.

Postura: o custo de um falso negativo é alto. Uma operação de lavagem que passa pela triagem vira exposição regulatória e reputacional, e o caso só volta à mesa quando já é tarde. Trate os sinais do dossiê como indícios a escalar, não como teses a provar. Na dúvida entre dois níveis, escolha o maior. Além do que está explícito, aponte red flags potenciais e hipóteses de tipologia compatíveis com o padrão observado, indicando em cada uma que informação adicional a confirmaria.

{ESQUEMA}"""

### Chamada, validação e métricas

In [203]:
from dotenv import load_dotenv
from openai import OpenAI
import time
import os

load_dotenv("../.env")  # caminho explícito: o notebook roda em nivel_1/ e o .env está na raiz

class LLMWrapper:
    """Encapsula a chamada à API. De fora só se usa call_openai(instrucao, dossie)."""

    NIVEIS_VALIDOS = {"baixo", "médio", "alto"}
    CAMPOS_OBRIGATORIOS = {"nivel_risco", "tipologia_suspeita", "red_flags", "justificativa"}

    def __init__(self, model="gpt-4o-mini", temperature=0):
        self.client = OpenAI(api_key=os.getenv("OPENAI_KEY"))
        self.model = model
        self.temperature = temperature  # 0 -> comparação entre prompts fica reproduzível

    def call_openai(self, instrucao, dossie):
        """Devolve (parecer, metricas, bruto). parecer é None quando a saída é recusada."""
        inicio = time.perf_counter()
        response = self.client.responses.create(
            model=self.model,
            temperature=self.temperature,
            instructions=instrucao,                      # o prompt (conservador ou arrojado)
            # a API exige a palavra "json" no proprio input quando o formato e json_object
            input=f"Dossie do cliente, responda em json:\n{json.dumps(dossie, ensure_ascii=False)}",
            text={"format": {"type": "json_object"}},
        )
        latencia = time.perf_counter() - inicio

        bruto = response.output_text
        parecer, erro = self._valida_estrutura(bruto)

        metricas = {
            "modelo": self.model,
            "latencia_s": round(latencia, 2),
            "tokens_entrada": response.usage.input_tokens,
            "tokens_saida": response.usage.output_tokens,
            "tokens_total": response.usage.total_tokens,
            "aceito": erro is None,
            "motivo_recusa": erro,
        }

        # Sem retentativa: saída malformada é recusada e devolvida crua para inspeção.
        # Com mais tempo, aqui entraria um loop de feedback devolvendo o erro ao modelo.
        return parecer, metricas, bruto

    def _valida_estrutura(self, bruto):
        """Devolve (parecer, erro). Com erro != None a saída é recusada."""
        try:
            d = json.loads(bruto)
        except json.JSONDecodeError as e:
            return None, f"não é JSON válido ({e})"
        if not isinstance(d, dict):
            return None, "JSON não é um objeto"

        # o conjunto de campos tem que bater exatamente: nem faltando, nem sobrando
        if faltando := self.CAMPOS_OBRIGATORIOS - d.keys():
            return None, f"campos ausentes: {sorted(faltando)}"
        if sobrando := d.keys() - self.CAMPOS_OBRIGATORIOS:
            return None, f"campos não previstos: {sorted(sobrando)}"

        nivel = d["nivel_risco"]
        if not isinstance(nivel, str) or nivel.strip().lower() not in self.NIVEIS_VALIDOS:
            return None, f"nivel_risco fora do domínio: {nivel!r}"
        d["nivel_risco"] = nivel.strip().lower()  # "Médio" -> "médio"

        if not isinstance(d["red_flags"], list):
            return None, "red_flags não é lista"
        if not all(isinstance(f, str) and f.strip() for f in d["red_flags"]):
            return None, "red_flags deve conter apenas strings não vazias"

        for campo in ("tipologia_suspeita", "justificativa"):
            if not isinstance(d[campo], str) or not d[campo].strip():
                return None, f"{campo} deve ser texto não vazio"

        return d, None

llm = LLMWrapper()

### Comparação prompts e resultados

In [204]:
resultados = {}
for nome, prompt in [("conservador", PROMPT_CONSERVADOR), ("arrojado", PROMPT_ARROJADO)]:
    parecer, metricas, bruto = llm.call_openai(prompt, dossie)
    resultados[nome] = {"parecer": parecer, "metricas": metricas, "bruto": bruto}

comparacao = pd.DataFrame(
    {
        nome: {
            **r["metricas"],
            "nivel_risco": r["parecer"]["nivel_risco"] if r["parecer"] else None,
            "n_red_flags": len(r["parecer"]["red_flags"]) if r["parecer"] else None,
        }
        for nome, r in resultados.items()
    }
).T

comparacao

,modelo,latencia_s,tokens_entrada,tokens_saida,tokens_total,aceito,motivo_recusa,nivel_risco,n_red_flags
conservador,gpt-4o-mini,12.92,762,137,899,True,None,médio,2
arrojado,gpt-4o-mini,2.09,779,181,960,True,None,alto,4


In [205]:
for nome, r in resultados.items():
    print(f"===== {nome.upper()} =====")
    if r["parecer"] is None:
        print(f"RECUSADO -> {r['metricas']['motivo_recusa']}")
        print(f"resposta crua:\n{r['bruto']}\n")
        continue
    print(json.dumps(r["parecer"], ensure_ascii=False, indent=2), "\n")

===== CONSERVADOR =====
{
  "nivel_risco": "médio",
  "tipologia_suspeita": "Fracionamento",
  "red_flags": [
    "Disparo da regra de fracionamento",
    "Três operações em um único dia com volume elevado"
  ],
  "justificativa": "O cliente apresentou um padrão de fracionamento, com três operações em um único dia que totalizaram um volume significativo. Embora a regra de valor atípico não tenha disparado, a concentração de operações e o volume total levantam preocupações. A análise deve ser aprofundada, mas não há evidências suficientes para classificar como alto risco."
} 

===== ARROJADO =====
{
  "nivel_risco": "alto",
  "tipologia_suspeita": "Fracionamento de operações",
  "red_flags": [
    "Três operações em um único dia com volume elevado",
    "Soma das operações no dia ultrapassa R$ 50.000",
    "Contraparte com histórico de operações suspeitas",
    "Mudança abrupta no padrão de operações"
  ],
  "justificativa": "O cliente apresentou um padrão de fracionamento com três oper

Mesmo dossiê, mesmo modelo, `temperature=0`. A única variável entre as duas chamadas é o prompt de postura, então a diferença é atribuível ao apetite de risco. Que é a intenção

**O que mudou.** O conservador classificou `médio` com 2 red flags e escreveu explicitamente que "não há evidências suficientes para classificar como alto risco". O arrojado classificou `alto` com 4 red flags. Custo e latência ficaram praticamente iguais (~900 vs ~965 tokens, ~4s nos dois), então a escolha entre os dois não é de custo — é de postura.

**O que o arrojado comprou e o que pagou.** Duas das quatro red flags dele não estão no dossiê: "contraparte com histórico de operações suspeitas" (não existe histórico de contraparte no que foi enviado) e "mudança abrupta no padrão de transações" (não há série histórica, só 4 operações). Pedir red flags *potenciais* fez o modelo preencher lacuna com suposição, tem mais propensão a **alucinar**. As 2 red flags do conservador são ambas rastreáveis ao dossiê.

**Leitura prática.** O conservador produz parecer auditável — cada afirmação tem lastro no dado enviado. O arrojado produz pista de investigação, que precisa ser confirmada antes de virar relatório. Para triagem em lote isso importa: o arrojado eleva o recall, mas parte do ganho é ruído que consome a hora de analista que a regra determinística tentou economizar.